[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/html_listing_scraper.ipynb)

# Scraping a listings page with BeautifulSoup

A local-services directory lists businesses as cards: name, locality, rating, services offered, response time. We download a saved copy of one such page, parse it with BeautifulSoup and turn each card into a row of a CSV.

## Setup

Install the parser if you don't already have it (uncomment the line below).

In [1]:
# !pip install requests beautifulsoup4 pandas lxml

In [2]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from copy import copy

## Download the page and save it locally

The page is a saved copy hosted with the course material, so it works the same on Colab and on your laptop. Saving the HTML once means we can re-run the parsing cells without downloading again.

In [3]:
url = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20collection/data/service_listings.html"
response = requests.get(url)
print(response.status_code)
if response.status_code == 200:
    with open("service_listings.html", "w", encoding="utf-8") as f:
        f.write(response.text)

200


## Parse the saved HTML

Open `service_listings.html` in a browser and *View Source* to see the structure we are about to walk.

In [4]:
with open('service_listings.html', 'r', encoding='utf-8') as file:
    html_content = file.read()
soup = bs(html_content, 'lxml')  # html.parser, lxml, html5lib

## Extract fields from each business card

For every `div.sk-card`: id, name, locality, lat/long, rating, list of services, response time and score.

In [5]:
cards = soup.find_all('div', class_="sk-card")  # this gets the whole div
print(len(cards), "cards")
servico_data=[]
business_info={}
for card in cards:
    # Id, name
    business_info['id']=card.get('businessid',"")
    business_info['name']=card.get('businessname',"")
    
    print("businessid",card.get('businessid',""))
    print("business name",card.get('businessname',""))
    #locality
    locality = card.find('div',class_="locality")
    #print(locality)
    location = locality.find('span')
    business_info['locality']=location.get_text()
    location=locality.find('div',class_='sk-link')
    if location:
        business_info['lat']=location.get('businesslat',"")
        business_info['long']=location.get('businesslong',"")
    else:
        business_info['lat']=''
        business_info['long']=''
    #Ratings
    rating_div=card.find('div','ratings-group')
    rating=rating_div.find('b')
    if rating:
        business_info['rating']=rating.get_text()
    else:
        business_info['rating']=''
    
    #list of services
    services_div=card.find('div',class_="tags-link")
    
    services_list=services_div.find_all('spam',class_="tag-link-item")
    services=[]
    if not services_list:
        services_list=services_div.find_all('span',class_="tag-link-item") 
    for service in services_list:
        service_name=service.get_text()
        if service_name not in services:
            services.append(service_name)
    
    #print(services_list)
    business_info['services']=services

    #keypoints
    key_points=card.find('div','key-points light')
    #print(key_points)
    key_point_span = key_points.find_all('span')
    for key_point in key_point_span:
        if 'Response Time' in key_point.get_text():
            b_tag=key_point.find('b')
            business_info['response_time']=b_tag.get_text()
        if 'Servico score' in key_point.get_text():
            b_tag=key_point.find('b')
            business_info['servico_score']=b_tag.get_text()


    servico_data.append(copy(business_info))
    #print(business_info)
    #exit()

10 cards
businessid 11771207
business name Mass Powertek
businessid 11530102
business name Star Cleaning Service
businessid 11140551
business name Akshay Builders
businessid 2494782
business name Entos De-Pest Solutions Pvt Ltd
businessid 10374974
business name Archana Transport & Travels
businessid 4775006
business name Aalampana Office chairs
businessid 11297371
business name My Crony
businessid 10948491
business name Hygiene Homes Marketing Pvt. Ltd.
businessid 10312017
business name one call done all home services
businessid 5240927
business name Sree vengateshwara home needs


## Save to CSV

In [6]:
df = pd.DataFrame(servico_data)
df.to_csv('service_listings.csv', index=None)
df

,id,name,locality,lat,long,rating,services,response_time,servico_score
0,11771207,Mass Powertek,"\nTambaram West, Chenn...",12.92563,80.1044,5.0/5,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,7.7
1,11530102,Star Cleaning Service,"\nPuzhal, Chennai, 600...",13.1673377,80.1921311,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,9
2,11140551,Akshay Builders,"\nMedavakkam, Chennai,...",,,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,9
3,2494782,Entos De-Pest Solutions Pvt Ltd,"\nVelachery, Chennai, ...",12.9780425,80.2211927,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,5.2
4,10374974,Archana Transport & Travels,"\nVelachery, Chennai, ...",12.9752364,80.2154474,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,4.3
5,4775006,Aalampana Office chairs,"\nTeynampet, Chennai, ...",12.922652,80.204284,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,4.3
6,11297371,My Crony,"\nAminjikarai, Chennai...",13.069373,80.220107,,"[Cleaning Services, Cobweb Cleaning, Bedroom C...",Within 15 Mins,6.9
7,10948491,Hygiene Homes Marketing Pvt. Ltd.,"\nSaidapet, Chennai, 6...",13.016498,80.229614,,"[Cleaning Services, Facade Cleaning, Glass Cle...",Within 15 Mins,4.7
8,10312017,one call done all home services,"\nKoyambedu, Chennai, ...",13.063739324057254,80.18554139882326,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,4.7
9,5240927,Sree vengateshwara home needs,"\nSaidapet, Chennai, 6...",13.02686,13.02686,,"[Cleaning Services, Bathroom Cleaning, Kitchen...",Within 15 Mins,4.7
